In [14]:
import sim          
import sympy as sp  
import numpy as np
import time
import math
import cv2
import matplotlib.pyplot as plt

def connect(port):
    sim.simxFinish(-1)
    clientID=sim.simxStart('127.0.0.1',port,True,True,2000,5) # Conectarse
    if clientID == 0: print("conectado a", port)
    else: print("no se pudo conectar")
    return clientID

In [15]:
clientID = connect(19999)

retCode,camara=sim.simxGetObjectHandle(clientID,'Vision_sensor',sim.simx_opmode_blocking)

retCode,ruedaDerecha=sim.simxGetObjectHandle(clientID,'RuedaR',sim.simx_opmode_blocking)
retCode,ruedaIzquierda=sim.simxGetObjectHandle(clientID,'RuedaL',sim.simx_opmode_blocking)

retCode,suction=sim.simxGetObjectHandle(clientID,'suctionPad',sim.simx_opmode_blocking)

ret,ultrasonidoDerecha=sim.simxGetObjectHandle(clientID,'SensorR',sim.simx_opmode_blocking)
ret,ultrasonidoIzquierda=sim.simxGetObjectHandle(clientID,'SensorL',sim.simx_opmode_blocking)
ret,ultrasonidoDelante=sim.simxGetObjectHandle(clientID,'SensorD',sim.simx_opmode_blocking)
ret,ultrasonidoAtras=sim.simxGetObjectHandle(clientID,'SensorA',sim.simx_opmode_blocking)
ret,cuerpo=sim.simxGetObjectHandle(clientID,'AWSD',sim.simx_opmode_blocking)

conectado a 19999


In [16]:
def setEffector(val):
# function that triggers the end effector remotely
# val is Int with value 0 or 1 to disable or activate the final actuator.
    res,retInts,retFloats,retStrings,retBuffer=sim.simxCallScriptFunction(clientID,
        "suctionPad", sim.sim_scripttype_childscript,"setEffector",[val],[],[],"", sim.simx_opmode_blocking)
    return res

def obtenerDistanciaSensor(ultrasonido):
    errorCode, detectionState, detectedPoint, detectedObjectHandle, detectedSurfaceNormalVector=sim.simxReadProximitySensor(clientID,ultrasonido, sim.simx_opmode_blocking)
    sensor_val=np.linalg.norm(detectedPoint)

    return sensor_val

def aplicarVelocidades(vel_izq, vel_der):
    # Pausar comunicacion para poder enviar los comandos al mismo tiempo
    sim.simxPauseCommunication(clientID, True)

    # Estos comandos se guardan en cola, no se ejecutan todavía
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, vel_izq, sim.simx_opmode_oneshot)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   vel_der, sim.simx_opmode_oneshot)

    # Enviar ambos comandos juntos para que arranquen a la vez
    sim.simxPauseCommunication(clientID, False)

def frenar():
    aplicarVelocidades(0, 0)
    sim.simxSynchronousTrigger(clientID) # Un paso físico para asegurar que se detiene en el simulador

In [17]:
"""def moverCasilla(v, distancia=0.24):
    # Obtener posicion inicial (eje X o Y segun orientacion)
    # Usamos streaming para inicializar
    ret, pos_inicial = sim.simxGetObjectPosition(clientID, cuerpo, -1, sim.simx_opmode_blocking)
    if ret != 0:
        print("Error al obtener posición inicial")
        return
    inicio_x = pos_inicial[0]   # Suponiendo que avanza en X
    inicio_y = pos_inicial[1]   # Suponiendo que avanza en Y
    
    # Arrancar motores (modo streaming)
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_streaming)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   v, sim.simx_opmode_streaming)
    
    # Activar streaming de la posicion porque si no, no puede ir leyendo la posicion actual
    sim.simxGetObjectPosition(clientID, cuerpo, -1, sim.simx_opmode_streaming)
    
    # Bucle de control
    while True:
        ret, pos_actual = sim.simxGetObjectPosition(clientID, cuerpo, -1, sim.simx_opmode_buffer)
        if ret == 0:   # Datos disponibles
            avanzado_x = pos_actual[0] - inicio_x
            avanzado_y = pos_actual[1] - inicio_y
            if abs(avanzado_x) >= distancia or abs(avanzado_y) >= distancia:
                break
        # Pequeña pausa para no saturar
        time.sleep(0.05)   # O time.sleep(0.05)
    
    # Detener ruedas, pero como la rueda izquierda para antes, genera un pequeño desvio
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, 0, sim.simx_opmode_streaming)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   0, sim.simx_opmode_streaming)
"""
def movimientoContinuo(velocidad, direccion):
    v = velocidad * direccion
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   v, sim.simx_opmode_blocking)

def moverCasilla(v_angular):
    v_lineal = v_angular*0.04
    t = 0.24/v_lineal
    t_real = t * 1.33

    aplicarVelocidades(v_angular, v_angular)
    time.sleep(t_real)
    frenar()
    

def giro90(v, direccion):
    radianes_rueda = math.radians(90 * direccion) * 0.17 / 0.08
    tiempo = radianes_rueda / v
    print(tiempo)
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,  -v, sim.simx_opmode_blocking)
    
    time.sleep(tiempo)
    
    # 5. Frenamos ambos motores
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,    0, sim.simx_opmode_blocking)

"""def girar(angulo_grados, vel_rad_s, tolerancia=0.5):
    # Obtener orientacion inicial
    ret, orientacion = sim.simxGetObjectOrientation(clientID, cuerpo, -1, sim.simx_opmode_blocking)
    if ret != 0:
        print("Error al leer orientación inicial")
        return
    angulo_inicial = round(orientacion[2])
    
    # Determinar sentido de giro y velocidad de las ruedas
    if angulo_grados > 0:
        # Giro horario
        vel_izq =  vel_rad_s
        vel_der = -vel_rad_s
    else:
        # Giro antihorario
        vel_izq = -vel_rad_s
        vel_der =  vel_rad_s
        
    angulo_objetivo_rad = angulo_inicial - math.radians(angulo_grados)
    
    # Normalizar objetivo al rango [-π, π]
    angulo_objetivo_rad = math.atan2(math.sin(angulo_objetivo_rad), math.cos(angulo_objetivo_rad))
    
    # Iniciar movimiento
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, vel_izq, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha, vel_der, sim.simx_opmode_blocking)
    
    # Configurar streaming de orientación
    sim.simxGetObjectOrientation(clientID, cuerpo, -1, sim.simx_opmode_streaming)
    
    tolerancia_rad = math.radians(tolerancia)
    while True:
        ret, orientacion = sim.simxGetObjectOrientation(clientID, cuerpo, -1, sim.simx_opmode_buffer)
        if ret == 0:
            angulo_actual = orientacion[2]
            # Diferencia angular minima (evita problemas con el cruce de ±π)
            error = angulo_objetivo_rad - angulo_actual
            error = math.atan2(math.sin(error), math.cos(error))
            if abs(error) < tolerancia_rad:
                break
        time.sleep(0.05)
"""

def foto():
    retCode, resolution, image=sim.simxGetVisionSensorImage(clientID,camara,0,sim.simx_opmode_oneshot_wait)
    img = np.array(image, dtype=np.float32)
    img = img.astype(np.uint8)  # Convierte a uint8
    img.resize(resolution[1], resolution[0], 3)
    img = cv2.flip(img, 0)
    #img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    plt.imshow(img)
    plt.show()

def girar(v_angular, grados):
    radio_rueda = 0.04  # 4 cm
    L = 0.115           # Distancia entre ruedas
    
    # Calcular la distancia que debe rodar cada rueda
    grados_abs = abs(grados)
    radianes_giro = math.radians(grados_abs)
    
    distancia_rueda = radianes_giro * L
    
    # Calcular velocidad lineal y tiempo 
    v_lineal = v_angular * radio_rueda
    t = distancia_rueda / v_lineal
    
    #Determinar el sentido de giro de cada rueda
    if v_angular > 0:
        vel_izq = v_angular  # Rueda izquierda hacia adelante
        vel_der = -v_angular   # Rueda derecha hacia atrás
    else:
        vel_izq = -v_angular   # Rueda izquierda hacia atrás
        vel_der = v_angular  # Rueda derecha hacia adelante

    aplicarVelocidades(vel_izq, vel_der)
    # Esperar a que complete el ángulo
    time.sleep(t) 
    frenar()
    
def giroContinuo(v, direccion):
    radianes_rueda = math.radians(90 * direccion) * 0.17 / 0.08
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   -v, sim.simx_opmode_blocking)

def detener():
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   0, sim.simx_opmode_blocking)


In [18]:
detener()
derecha = obtenerDistanciaSensor(ultrasonidoDerecha)
izquierda = obtenerDistanciaSensor(ultrasonidoIzquierda)
delante = obtenerDistanciaSensor(ultrasonidoDelante)
atras = obtenerDistanciaSensor(ultrasonidoAtras)

print(derecha)
print(izquierda)

0.055000957173889024
0.05499864002899779


In [19]:
#moverCasilla(1)
moverCasilla(1)
time.sleep(0.25)
#moverCasilla(1)
#time.sleep(0.25)
#girar(1, 90)
#foto()
#time.sleep(0.25)
#moverCasilla(1)